# **Fundamentos de RL**
## **Regresión Logística**

### Librerías

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

### Cargar dataset


Diabetics prediction using logistic regression

https://www.kaggle.com/datasets/kandij/diabetes-dataset

In [46]:
data = pd.read_csv(filepath_or_buffer='https://raw.githubusercontent.com/cdavid2804/test-1/main/diabetes2.csv')
print("Número de datos originales:",np.shape(data))
# Eliminar datos que tienen "nan"
data = data.dropna()
print("Número de datos filtrado:",np.shape(data))
data.head(10)

Número de datos originales: (768, 9)
Número de datos filtrado: (768, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1
5,5,116,74,0,0,25.6,0.201,30,0
6,3,78,50,32,88,31.0,0.248,26,1
7,10,115,0,0,0,35.3,0.134,29,0
8,2,197,70,45,543,30.5,0.158,53,1
9,8,125,96,0,0,0.0,0.232,54,1


## **Algoritmo**

### Matriz de correlación entre variables

In [47]:
data.corr()# data conunto de datos donde se almacena el dataset, corr es la matriz de correlación. valor mayor a 1 significa mas correlacion

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
Pregnancies,1.000000,0.129459,0.141282,-0.081672,-0.073535,0.017683,-0.033523,0.544341,0.221898
Glucose,0.129459,1.000000,0.152590,0.057328,0.331357,0.221071,0.137337,0.263514,0.466581
BloodPressure,0.141282,0.152590,1.000000,0.207371,0.088933,0.281805,0.041265,0.239528,0.065068
SkinThickness,-0.081672,0.057328,0.207371,1.000000,0.436783,0.392573,0.183928,-0.113970,0.074752
Insulin,-0.073535,0.331357,0.088933,0.436783,1.000000,0.197859,0.185071,-0.042163,0.130548
BMI,0.017683,0.221071,0.281805,0.392573,0.197859,1.000000,0.140647,0.036242,0.292695
DiabetesPedigreeFunction,-0.033523,0.137337,0.041265,0.183928,0.185071,0.140647,1.000000,0.033561,0.173844
Age,0.544341,0.263514,0.239528,-0.113970,-0.042163,0.036242,0.033561,1.000000,0.238356
Outcome,0.221898,0.466581,0.065068,0.074752,0.130548,0.292695,0.173844,0.238356,1.000000


### División de los datos (entrenamiento y prueba)
#### Agregar una vector columna de "$1$" a $X$ (coeficiente de intersección)

$$X=\begin{bmatrix}
x_{11} & x_{12} & \cdots & x_{1n} \\
x_{21} & x_{22} & \cdots & x_{2n} \\
\vdots & \vdots & \ddots & \vdots \\
x_{m1} & x_{m2} & \cdots & x_{mn}  
\end{bmatrix}\rightarrow X=\begin{bmatrix}
1 & x_{11} & x_{12} & \cdots & x_{1n} \\
1 & x_{21} & x_{22} & \cdots & x_{2n} \\
\vdots & \vdots & \vdots & \ddots & \vdots \\
1 & x_{m1} & x_{m2} & \cdots & x_{mn}  
\end{bmatrix}$$

In [35]:
# caracteristicas extraidas de forma de matriz convertida en arreglo
X = data[['Pregnancies','Glucose','Insulin','BloodPressure','SkinThickness']]
y = data['Outcome']
X = X.values# almacena valor de entrada
y = y.values# almacena valor de salida

### Normalización de los datos
$$ X_{norm} = \frac{X - \mu}{\sigma} $$

- $\mu$: media
- $\sigma$: desviación estándar

$$ X_{norm} = \frac{X - X_{min}}{X_{max}-X_{min}} $$

- $X_{max}$: valor máximo
- $X_{min}$: valor mínimo

In [36]:
X_norm = (X - np.mean(X, axis = 0)) / np.std(X, axis = 0)
#np.mean saca media y np.std saca desviacion estandar
X_norm = np.hstack((np.ones((X_norm.shape[0],1)), X_norm))

#Division de datos
X_train, X_test, y_train, y_test = train_test_split(X_norm, y, test_size = 0.15)
print(f"Training data: {np.shape(X_train)}")
print(f"Test data: {np.shape(X_test)}")

Training data: (652, 6)
Test data: (116, 6)


### Definir la función sigmoidea

$$\sigma(z)=\displaystyle\frac{1}{1+e^{-z}}$$

In [37]:
def sigmoid(z):
  s = 1 / (1 + np.exp(-z))# np.exp para colocar exponencial euler
  return s

### Definir la función de pérdida

$$loss = -\displaystyle\frac{1}{m}\displaystyle\sum_{i=1}^m\left[y\ln(h_{\theta})+(1-y)\ln(1-h_{\theta})\right] $$

$$h_{\theta} = \sigma(z)$$

In [38]:
def loss(h, y):
  lss = (-(y*np.log(h) + (1-y) * np.log(1 - h))).mean()
  return lss

### Descenso del gradiente

$$
\begin{align}
&\begin{array}{l}
\hline
\textbf{Algorithm:}\text{ Gradient Descent for Logistic Regression } \\
\hline
\textbf{Input:} \\
1: \hspace{3mm} \text{Train set} {(X_1,y_1),...,(X_n,y_n)},  \\
2: \hspace{3mm} \text{Learning rate } \alpha > 0, \\
3: \hspace{3mm} \text{Number of iterations T},  \\
4: \hspace{3mm} \text{Initial weights }\theta  \\
5: \hspace{3mm} \textbf{for } t=1,2,...,T \textbf{ do}\\
6: \hspace{10mm} \text{Calcualte value of objective function: } z=X.\theta \\
7: \hspace{10mm} \text{Calcualte sigmoid function: } h_{\theta}=\sigma(z) \\
8: \hspace{10mm} \text{Compute gradients: } \triangledown_{\theta}J(\theta)=\displaystyle\frac{1}{m}X^{T}(h_{\theta}-y) \\
9: \hspace{10mm} \text{Update weights: } \theta_{update} =\theta - \alpha\triangledown_{\theta}J(\theta) \\
10: \hspace{2mm} \textbf{end for} \\
11: \hspace{2mm} \textbf{return} \text{ Output: extract weights } \theta \\
\hline
\end{array}
\end{align}
$$

In [39]:
def logistic_regression(X, y, alpha, t):
  theta = np.random.rand(X.shape[1])

  for i in range(t):
    m = y.size
    z = np.dot(X, theta)
    h = sigmoid(z)
    gradient = np.dot(X.T, (h-y)) / m
    theta = theta - alpha * gradient

    if (i%100) == 0:
      print(f"Iteration {i}, Loss: {loss(h, y):.4f}")

  return theta

alpha = 0.001
num_iterations = 100000
np.random.seed(23)
theta = logistic_regression(X_train, y_train, alpha, num_iterations)

Iteration 0, Loss: 0.7403
Iteration 100, Loss: 0.7332
Iteration 200, Loss: 0.7262
Iteration 300, Loss: 0.7195
Iteration 400, Loss: 0.7129
Iteration 500, Loss: 0.7066
Iteration 600, Loss: 0.7004
Iteration 700, Loss: 0.6944
Iteration 800, Loss: 0.6885
Iteration 900, Loss: 0.6829
Iteration 1000, Loss: 0.6774
Iteration 1100, Loss: 0.6720
Iteration 1200, Loss: 0.6668
Iteration 1300, Loss: 0.6618
Iteration 1400, Loss: 0.6569
Iteration 1500, Loss: 0.6522
Iteration 1600, Loss: 0.6476
Iteration 1700, Loss: 0.6431
Iteration 1800, Loss: 0.6388
Iteration 1900, Loss: 0.6346
Iteration 2000, Loss: 0.6305
Iteration 2100, Loss: 0.6266
Iteration 2200, Loss: 0.6228
Iteration 2300, Loss: 0.6191
Iteration 2400, Loss: 0.6155
Iteration 2500, Loss: 0.6121
Iteration 2600, Loss: 0.6087
Iteration 2700, Loss: 0.6055
Iteration 2800, Loss: 0.6023
Iteration 2900, Loss: 0.5993
Iteration 3000, Loss: 0.5964
Iteration 3100, Loss: 0.5935
Iteration 3200, Loss: 0.5908
Iteration 3300, Loss: 0.5881
Iteration 3400, Loss: 0.58

### Hacer predicciones

**Límite de decisión**

$$
\sigma(X.\theta) \geq  0.5
\rightarrow
y_{pred} = 1
$$
$$
\sigma(X.\theta) < 0.5
\rightarrow
y_{pred} = 0
$$

In [40]:
def predict(X, theta):
  umbral = 0.5
  z = sigmoid(np.dot(X, theta))
  y = np.where(z >= umbral, 1, 0)
  return y
y_pred = predict(X_test, theta)
print(f"Predicciones: {y_pred}")
print(f"Real: {y_test}")

Predicciones: [0 1 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 1 0 1 0 0 0 0 0 1 0 0 0 1 0 1 0 0 0
 0 0 0 1 0 1 0 0 0 0 0 0 0 0 1 1 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0
 0 1 0 1 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 1 0 0 0 0 0 1 1 0 0 0 0 0 0 1 1 0
 1 0 0 0 0]
Real: [0 1 0 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 1 0 1 0 0 0 1 0 1 0 1 1 1 0 0 0
 1 0 1 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 1 1 0 0 0 1 0 0 0 1 0 0 0 1 1 0 1 0 0
 0 1 0 1 1 0 0 1 0 0 0 0 1 0 1 1 0 1 0 0 1 1 0 0 1 1 1 1 0 1 1 0 0 0 1 1 0
 1 0 0 0 0]


### Evaluación del modelo

**Matriz de confusión**

$$
\text{Confusion matrix}$$
$$
\begin{array}{|c|c|c|}
\hline
&\textbf{Positive} & \textbf{Negative} \\
\hline
\textbf{Positive} & TP & FP\\
\hline
\textbf{Negative} & FN & TN \\
\hline
\end{array}
$$

$TP: \text{ True positive}$

$TN: \text{ True negative}$

$FP: \text{ False positive}$

$FN: \text{ False negative}$

In [41]:
def confusssion_matriz(y_read, y_pred):
  true_positive = 0
  true_negative = 0
  false_positive = 0
  false_negative = 0
  #len(y_read) calcula dimension de datos de y_read
  for i in range(len(y_read)):
    if y_read[i] == 1 and y_pred[i] == 1:
      true_positive += 1
    if y_read[i] == 0 and y_pred[i] == 0:
      true_negative += 1
    if y_read[i] == 1 and y_pred[i] == 0:
      false_negative += 1
    if y_read[i] == 0 and y_pred[i] == 1:
      false_positive += 1
  table = np.array(([[true_positive, false_positive], [false_negative, true_negative]]))

  return table

In [42]:
matrixConf = confusssion_matriz(y_test, y_pred)
matrixConf_df = pd.DataFrame(matrixConf, columns = ["Positive","Negative"], index = ["Positive","Negative"])
matrixConf_df

,Positive,Negative
Positive,18,4
Negative,22,72


*Accuracy*

$$
Accuracy=\frac{TP+TN}{TP+FP+TN+FN}
$$

In [48]:
#accuracy mide cuantos aciertos tiene el modelo con respecto a los valores reales
tp = matrixConf[0,0]
fp = matrixConf[0,1]
fn = matrixConf[1,0]
tn = matrixConf[1,1]


accuracy = (tp + tn) / (tp + fp + tn + fn)
print(f"Accuracy: {accuracy:.5f}")

Accuracy: 0.77586


*Precision*

$$
Precision=\frac{TP}{TP+FP}
$$

In [44]:
precision = tp / (tp + fp)
print(f"Precision: {precision:.5f}")

Precision: 0.81818


*Recall (sensibilidad o tasa de verdaderos positivos)*

$$
Recall=\frac{TP}{TP+FN}
$$

In [45]:
recall = tp /(tp + fn)
print(f"Recall: {recall:.5f}")

Recall: 0.45000
